In [ ]:
import os
import asyncio
import datetime

from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

from typing import TypedDict, List

In [ ]:
load_dotenv()

GroqApiKey = os.getenv("GroqAPI")
ModelName = "llama-3.3-70b-versatile"
EmbeddingModel = "sentence-transformers/all-MiniLM-L6-v2"
SessionLogDir = "session_logs"

if not GroqApiKey:
    raise ValueError("GroqAPI not found in .env")

print(f"Model: {ModelName}")
print(f"Embeddings: {EmbeddingModel}")
print(f"Log Dir: {SessionLogDir}")

In [ ]:
RawDocuments = [
    """LangGraph is a library for building stateful, multi-actor applications with LLMs.
    It extends LangChain to support cycles and persistence. Graphs are defined with nodes
    (Python functions) and edges (transitions between nodes). State is shared across all nodes.""",

    """Streaming in LangGraph allows receiving output token-by-token or node-by-node.
    graph.stream() supports three modes: 'values' returns full state after each node,
    'updates' returns only the delta, and 'debug' returns verbose internal events.
    astream_events() enables token-level streaming by firing on_chat_model_stream events.""",

    """MemorySaver is LangGraph's built-in in-memory checkpointer. It stores conversation
    state between turns using a thread_id. Each unique thread_id is an isolated conversation.
    Combine MemorySaver with astream_events() to get both persistence and streaming.""",

    """RAG stands for Retrieval-Augmented Generation. It grounds LLM responses in external
    documents by first retrieving relevant chunks from a vector store, then passing them
    as context to the LLM. FAISS is a popular in-memory vector store for local RAG setups.""",

    """Groq is an AI inference platform offering extremely fast LLM inference.
    LLaMA-3.3-70b-versatile is a capable open-source model hosted on Groq.
    It supports function calling, streaming, and long context windows up to 128k tokens.""",
]

print(f"Loaded {len(RawDocuments)} raw documents.")

In [ ]:
def buildVectorStore(RawDocs, EmbedModelName):

    Documents = [Document(page_content=Text) for Text in RawDocs]

    Splitter = RecursiveCharacterTextSplitter(
        chunk_size = 300,
        chunk_overlap = 50
    )
    Chunks = Splitter.split_documents(Documents)

    Embeddings = HuggingFaceEmbeddings(model_name=EmbedModelName)

    VectorStore = FAISS.from_documents(Chunks, Embeddings)

    print(f"Vector store built with {len(Chunks)} chunks.")
    return VectorStore

VectorStore = buildVectorStore(RawDocuments, EmbeddingModel)

In [ ]:
class GraphState(TypedDict):
    Question : str           
    Context : str           
    Answer : str           
    ChatHistory : List[dict]    

LLM = ChatGroq(
    api_key = GroqApiKey,
    model = ModelName,
    streaming = True,      
    temperature = 0.3,
)

In [ ]:
async def retrieve(State: GraphState) -> dict:
   
    Question = State["Question"]

    RelevantDocs = VectorStore.similarity_search(Question, k=3)
    
    Context = "\n\n".join([Doc.page_content for Doc in RelevantDocs])

    return {"Context": Context}

async def generate(State: GraphState) -> dict:

    Question = State["Question"]
    Context = State["Context"]
    ChatHistory = State.get("ChatHistory", [])

    Messages = []

    SystemPrompt = (
        "You are a helpful assistant. Use the context below to answer the user's question. If the context does not contain the answer, say so honestly.\n\n"
        f"Context:\n{Context}"
    )
    Messages.append({"role": "system", "content": SystemPrompt})

    for Turn in ChatHistory:
        Messages.append({"role": Turn["role"], "content": Turn["content"]})

    Messages.append({"role": "user", "content": Question})
    
    Response = await LLM.ainvoke(Messages)
    Answer = Response.content

    UpdatedHistory = ChatHistory + [ {"role": "user", "content": Question}, {"role": "assistant", "content": Answer}, ]

    return {"Answer": Answer, "ChatHistory": UpdatedHistory}

In [ ]:
def buildGraph():
    
    Builder = StateGraph(GraphState)
    Memory = MemorySaver()      

    Builder.add_node("retrieve", retrieve)
    Builder.add_node("generate", generate)

    Builder.add_edge(START,      "retrieve")
    Builder.add_edge("retrieve", "generate")
    Builder.add_edge("generate", END)
    
    Graph = Builder.compile(checkpointer=Memory)

    return Graph


Graph = buildGraph()

In [ ]:
def createSessionLogger(LogDir):

    os.makedirs(LogDir, exist_ok=True)

    Timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    LogFilePath = os.path.join(LogDir, f"session_{Timestamp}.txt")

    with open(LogFilePath, "w", encoding="utf-8") as F:
        F.write(f"\tRAG Session Log\n")
        F.write(f"Started: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        F.write(f"Model: {ModelName}\n")
        F.write("\n\n")

    print(f"Session log: {LogFilePath}")
    return LogFilePath


def logTurn(LogFilePath, Question, Answer):
   
    Timestamp = datetime.datetime.now().strftime("%H:%M:%S")

    with open(LogFilePath, "a", encoding="utf-8") as F:
        F.write(f"[{Timestamp}] User: {Question}\n")
        F.write(f"[{Timestamp}] AI : {Answer}\n")
        F.write("\n")

In [ ]:
async def streamQuery(UserQuestion, ThreadId, LogFilePath):
    
    InputState = {"Question": UserQuestion}
    Config = {"configurable": {"thread_id": ThreadId}}

    FullAnswer = ""
    
    CurrentNode = None

    async for Event in Graph.astream_events(InputState, config=Config, version="v2"):

        EventType = Event["event"]
        EventName = Event.get("name", "")

        if EventType == "on_chain_start" and EventName in ("retrieve", "generate"):

            if EventName != CurrentNode:
                CurrentNode = EventName

                NodeLabels = {
                    "retrieve": "[Retrieving]",
                    "generate": "[Generating]",
                }
                Label = NodeLabels.get(EventName, f"[{EventName}]")
                print(f"\n{Label}", flush=True)

        elif EventType == "on_chat_model_stream":

            Chunk = Event.get("data", {}).get("chunk")

            if Chunk is not None:
                TokenText = getattr(Chunk, "content", "")

                if TokenText:
                    print(TokenText, end="", flush=True)
                    FullAnswer += TokenText

    
    print("\n")

    if FullAnswer.strip():
        logTurn(LogFilePath, UserQuestion, FullAnswer)

    return FullAnswer

In [ ]:
async def runChatLoop():
    
    print("\n")
    print("Live Streaming RAG Terminal")
    print("Type 'quit' or 'exit' to stop")
    print("\n")

    LogFilePath = createSessionLogger(SessionLogDir)

    ThreadId = "rag-session-001"

    TurnNumber = 0

    while True:
        try:
            UserInput = input("\nYou: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n\nSession ended by user.")
            break

        if UserInput.lower() in ("quit", "exit", "q"):
            print("\nGoodbye! Session log saved to:", LogFilePath)
            break

        if not UserInput:
            print("(Empty input, please type a question.)")
            continue

        TurnNumber += 1
        print(f"\nAssistant:", end=" ", flush=True)

        try:
            await streamQuery(UserInput, ThreadId, LogFilePath)

        except Exception as Error:
            print(f"\n[Error during streaming: {Error}]")
            print("Please try again.")

In [ ]:
async def runSingleTest():
    
    LogFilePath = createSessionLogger(SessionLogDir)
    TestQuestion = "What is LangGraph and how does streaming work in it?"
    ThreadId = "test-thread-001"

    print(f"Question: {TestQuestion}")
    print("Assistant:", end=" ", flush=True)

    await streamQuery(TestQuestion, ThreadId, LogFilePath)

    print(f"\nLog saved to: {LogFilePath}")

await runSingleTest()

In [ ]:
await runChatLoop()

In [ ]:
def showLatestLog(LogDir):
   
    if not os.path.exists(LogDir):
        print("No logs found yet.")
        return

    LogFiles = sorted( [F for F in os.listdir(LogDir) if F.startswith("session_")] )

    if not LogFiles:
        print("No session logs found.")
        return

    LatestLog = os.path.join(LogDir, LogFiles[-1])
    print(f"Reading: {LatestLog}\n")

    with open(LatestLog, "r", encoding="utf-8") as F:
        print(F.read())

showLatestLog(SessionLogDir)